# FLUX.1 [schnell] — free image server for BibleMusically

Serves the **Apache-2.0 licensed [FLUX.1 schnell](https://huggingface.co/black-forest-labs/FLUX.1-schnell)** image model (commercially usable, YouTube-safe) on a **free Kaggle/Colab GPU**, exposing a small REST API through a public **cloudflared** tunnel. This is the free open-source alternative to Midjourney.

Copy the printed `https://xxxx.trycloudflare.com` URL into the desktop app: **Settings → Image engine → FLUX → FLUX server URL**, then set the engine to *FLUX*.

**Before you run:** enable the GPU — Kaggle: *Settings → Accelerator → GPU T4 x2*; Colab: *Runtime → Change runtime type → GPU*.

> FLUX.1 schnell needs ~16 GB VRAM in bf16; a single Kaggle T4 (16 GB) works with CPU offload enabled (set below). Free tunnel URLs and Kaggle sessions are temporary — re-run and re-paste when the URL stops working.

## 1. Install dependencies

In [ ]:
import subprocess, sys

# FLUX needs diffusers + transformers + accelerate. If Internet is OFF this cell
# can't reach PyPI ("Could not resolve host") — enable it in Session options first.
# On Colab/Kaggle pip may print "dependency conflicts" against pre-installed base
# packages (google-colab, moviepy, gym, tpot, pandas...) we never import — non-fatal.
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers>=0.32','transformers','accelerate','sentencepiece',
                'protobuf','fastapi','uvicorn'], check=True)

# ---- Verify the environment actually works (this, not pip's red text, is what matters) ----
import importlib
ok = True
for mod in ('torch','diffusers','transformers'):
    try:
        m = importlib.import_module(mod)
        print(f'  ✅ {mod:12s} {getattr(m,"__version__","ok")}')
    except Exception as ex:
        ok = False
        print(f'  ❌ {mod:12s} FAILED: {ex}')
print('\ndeps installed — environment OK.' if ok
      else '\n⚠️  An import FAILED above — fix that before continuing.')


## 2. Install cloudflared (public tunnel, no signup)

In [ ]:
import subprocess
if subprocess.run(['which', 'cloudflared'], capture_output=True).returncode != 0:
    subprocess.run(
        'curl -L --output /tmp/cloudflared.deb '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb '
        '&& dpkg -i /tmp/cloudflared.deb',
        shell=True, check=True)
print('cloudflared ready.')

## 3. Hugging Face access (required — FLUX.1 schnell is a gated repo)

Even though FLUX.1 schnell is Apache-2.0, black-forest-labs gates the HF repo behind a license
click-through, so downloading it needs an **authenticated** Hugging Face token that has accepted
that license — an anonymous/unauthenticated download gets a `401 GatedRepoError`, which is exactly
what used to happen here.

One-time setup (do this before running the next cell):
1. Visit <https://huggingface.co/black-forest-labs/FLUX.1-schnell> while logged into HF and click
   **"Agree and access repository"**.
2. Create a **read-scope** access token at <https://huggingface.co/settings/tokens>.
3. In this notebook: **Add-ons → Secrets → Add a new secret**, name it `HF_TOKEN`, paste the token,
   and make sure it's attached/enabled for this notebook.

## 4. Load FLUX.1 [schnell]

First run downloads ~24 GB of weights (a few minutes). `enable_model_cpu_offload` keeps it within a single 16 GB T4.

In [ ]:
import torch
from diffusers import FluxPipeline

# FLUX.1-schnell is gated on HF — resolve a token from Kaggle secrets first (the normal path),
# falling back to an HF_TOKEN env var (Colab / local runs, where secrets aren't a thing). Fails
# fast with clear instructions instead of the opaque 401 GatedRepoError from from_pretrained().
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
if not hf_token:
    import os
    hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'No HF_TOKEN found. FLUX.1-schnell is a gated HF repo: accept the license at '
        'https://huggingface.co/black-forest-labs/FLUX.1-schnell, create a read-scope token at '
        'https://huggingface.co/settings/tokens, then add it as a Kaggle secret named HF_TOKEN '
        '(Add-ons -> Secrets in this notebook) and re-run this cell.'
    )

pipe = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16, token=hf_token)
pipe.enable_model_cpu_offload()  # comment out if you have >=24 GB VRAM and want max speed
print('FLUX.1 schnell loaded.')

## 5. (Optional) set an API key

Leave blank for the simplest setup. If set, paste the same value into the app's *API key* field.

In [ ]:
API_KEY = ''  # e.g. 'my-secret-key' — must match the app's FLUX API key field
print('API key set.' if API_KEY else 'No API key (open server).')

## 6. Launch the REST server + tunnel

Exposes `POST /generate` (what the app calls), `GET /images/<id>.png`, and `GET /health`. Keep this cell running — closing it stops the server.

In [ ]:
import os, io, re, time, uuid, threading, subprocess
import torch
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse, FileResponse
import uvicorn

PORT = 8002
IMG_DIR = '/tmp/flux_images'
os.makedirs(IMG_DIR, exist_ok=True)
app = FastAPI()

def _check_auth(request: Request):
    if not API_KEY:
        return
    auth = request.headers.get('authorization', '')
    if auth != f'Bearer {API_KEY}':
        raise HTTPException(status_code=401, detail='bad api key')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'FLUX.1-schnell'}

@app.post('/generate')
async def generate(request: Request):
    _check_auth(request)
    body = await request.json()
    prompt = (body.get('prompt') or '').strip()
    if not prompt:
        raise HTTPException(status_code=400, detail='prompt required')
    n = int(body.get('num_images', 4))
    steps = int(body.get('steps', 4))          # schnell is optimized for ~4 steps
    width = int(body.get('width', 1024))
    height = int(body.get('height', 1024))
    paths = []
    for _ in range(max(1, min(n, 8))):
        image = pipe(
            prompt,
            num_inference_steps=steps,
            guidance_scale=0.0,                 # schnell ignores guidance
            width=width, height=height,
            generator=torch.Generator('cpu').manual_seed(int.from_bytes(os.urandom(4), 'big')),
        ).images[0]
        fid = f'{uuid.uuid4().hex}.png'
        image.save(os.path.join(IMG_DIR, fid))
        paths.append(f'/images/{fid}')
    return JSONResponse({'images': paths})

@app.get('/images/{fid}')
def get_image(fid: str):
    if not re.fullmatch(r'[0-9a-f]+\.png', fid):
        raise HTTPException(status_code=400, detail='bad id')
    fp = os.path.join(IMG_DIR, fid)
    if not os.path.isfile(fp):
        raise HTTPException(status_code=404, detail='not found')
    return FileResponse(fp, media_type='image/png')

# Run uvicorn in a background thread so we can also manage the tunnel from this cell.
def _serve():
    uvicorn.run(app, host='127.0.0.1', port=PORT, log_level='warning')
threading.Thread(target=_serve, daemon=True).start()

import urllib.request
print('Waiting for the image server to come up...')
for _ in range(60):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=3)
        print('Server is up.')
        break
    except Exception:
        time.sleep(2)

# Open the public tunnel and print its URL.

# ── Batch-run guard v2 ──────────────────────────────────────────────
# Source-update pushes run GPU-less and should exit fast; a GPU batch run is a
# DELIBERATE server start (the app's "Start server" button pushes with GPU on)
# and must open the tunnel and keep serving.
_is_batch = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive').lower() == 'batch'

# Two independent questions, asked separately because they fail apart — and which one failed IS
# the diagnosis:
#   nvidia-smi  — does this container have a GPU and a working driver at all?
#   torch.cuda  — can the framework that runs the model actually reach it?
# A GPU-off source-update push answers no to both, and so does Kaggle declining an accelerator.
# An install step that replaced Kaggle's CUDA torch with a CPU-only wheel answers YES to the
# first and no to the second. The old guard asked only nvidia-smi and reported every "no" as an
# exhausted weekly quota — which sent people to a quota page that had 29.8 of 30 hours left on it.
_smi_rc, _smi_note = None, ''
try:
    _smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=120)
    _smi_rc = _smi.returncode
    _smi_note = (_smi.stderr or '').strip().replace('\n', ' ')[:200]
except FileNotFoundError:
    _smi_note = 'nvidia-smi is not installed on this container'
except Exception as _ex:
    _smi_note = '{}: {}'.format(type(_ex).__name__, _ex)
try:
    import torch as _t
    _torch_cuda, _torch_ver = bool(_t.cuda.is_available()), _t.__version__
except Exception as _ex:
    _torch_cuda, _torch_ver = False, 'unavailable ({})'.format(type(_ex).__name__)

# Serving is gated on torch rather than on the driver, because the model is loaded onto whatever
# device torch reports. A container that has a GPU torch cannot see would otherwise open a public
# tunnel to a server generating on CPU — minutes of audio at hours of wall clock, which is a worse
# outcome than not starting, and much harder to diagnose from the app.
_has_gpu = _torch_cuda
if _is_batch and not _has_gpu:
    print('=' * 70)
    print('  NO GPU ON THIS RUN — not serving.')
    print('  nvidia-smi: ' + ('exit {}'.format(_smi_rc) if _smi_rc is not None else 'did not run')
          + (' — {}'.format(_smi_note) if _smi_note else ''))
    print('  torch {}: cuda.is_available() = {}'.format(_torch_ver, _torch_cuda))
    print('  CUDA_VISIBLE_DEVICES = {!r}'.format(os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')))
    if _smi_rc == 0:
        print('  GPU PRESENT BUT TORCH CANNOT USE IT — an install step in this notebook replaced')
        print("  Kaggle's CUDA build of torch with a CPU-only one. Fix that cell; the quota is")
        print('  not the problem here.')
    else:
        print('  KAGGLE GAVE THIS SESSION NO ACCELERATOR. The app always asks for one, so this is')
        print('  the scheduler declining: the weekly quota is spent, both GPU session slots are')
        print('  busy, or no GPU was free at that moment. That last case is common and transient —')
        print('  if the quota page still shows hours left, simply start again.')
        print('  Quota: https://www.kaggle.com/settings  (Accelerator usage).')
    print('  (A deliberate GPU-off push is just a cheap source update - nothing is wrong.)')
    print('=' * 70)
    print('Start the server from the app (Start server button) or run interactively with GPU on.')
else:
    import urllib.request, urllib.error

    # ── Reliable public tunnel with self-healing ───────────────────────────────
    # The failure this fixes: cloudflared prints a *.trycloudflare.com URL and even registers an
    # edge connection, yet the Cloudflare edge never actually ROUTES the hostname, so the URL never
    # answers and the app times out. Quick tunnels are flaky per-process and QUIC (UDP) egress can
    # be throttled. So we: (1) probe our OWN public URL to confirm it truly routes, (2) auto-restart
    # cloudflared — first over QUIC, then over HTTP/2, which survives UDP throttling — and (3) fall
    # back to localhost.run (ssh) if cloudflared keeps failing. Only a URL that actually ANSWERS is
    # printed as ready, and every step is logged so a failure is diagnosable from the app's log tail.
    _url_re = re.compile(r'https://[-a-z0-9]+\.(?:trycloudflare\.com|lhr\.life|serveo\.net)')

    def _probe_public(url, timeout=8):
        # True iff the tunnel truly routes: ANY HTTP status < 500 back proves the edge reached our
        # server. A connection error/timeout, or Cloudflare's own 5xx (e.g. 530 = tunnel down),
        # means "not routed yet".
        try:
            with urllib.request.urlopen(url.rstrip('/') + '/', timeout=timeout) as r:
                return r.status < 500
        except urllib.error.HTTPError as he:
            return he.code < 500
        except Exception:
            return False

    def _pump(proc, holder, tag='tunnel'):
        def _run():
            for line in proc.stdout:
                print(f'[{tag}] {line}', end='')
                m = _url_re.search(line)
                if m and not holder.get('url'):
                    holder['url'] = m.group(0)
        threading.Thread(target=_run, daemon=True).start()

    def _spawn_cloudflared(protocol):
        args = ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}']
        if protocol:
            args += ['--protocol', protocol]
        print(f'[tunnel] launching cloudflared (protocol={protocol or "auto"})...', flush=True)
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _spawn_localhostrun():
        print('[tunnel] launching localhost.run over ssh...', flush=True)
        p = subprocess.Popen(
            ['ssh', '-o', 'StrictHostKeyChecking=no', '-o', 'UserKnownHostsFile=/dev/null',
             '-o', 'ServerAliveInterval=30', '-R', f'80:localhost:{PORT}', 'nokey@localhost.run'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        h = {}; _pump(p, h); return p, h

    def _bring_up(proc, holder, url_wait=30, route_wait=75):
        t0 = time.time()
        while time.time() - t0 < url_wait and not holder.get('url'):
            if proc.poll() is not None:
                print('[tunnel] process exited before printing a URL.', flush=True); return None
            time.sleep(1)
        url = holder.get('url')
        if not url:
            print(f'[tunnel] no URL within {url_wait}s.', flush=True); return None
        print(f'Tunnel URL: {url}', flush=True)
        print('Waiting for the edge to route it...', flush=True)
        t1 = time.time()
        while time.time() - t1 < route_wait:
            if proc.poll() is not None:
                print('[tunnel] tunnel process exited during the routing wait.', flush=True); return None
            if _probe_public(url):
                print(f'[tunnel] OK: answered from inside Kaggle after {int(time.time()-t1)}s (a brand-new address can still take minutes to route elsewhere).', flush=True)
                return url
            time.sleep(4)
        print(f'[tunnel] {url} never answered within {route_wait}s - treating as dead.', flush=True)
        return None

    _attempts = [('cf', 'quic'), ('cf', 'http2'), ('lhr', None)]
    active_proc = None; public_url = None
    for _i, (_kind, _proto) in enumerate(_attempts, 1):
        print(f'\n[tunnel] ===== attempt {_i}/{len(_attempts)}: {_kind} {_proto or ""} =====', flush=True)
        try:
            _p, _h = _spawn_cloudflared(_proto) if _kind == 'cf' else _spawn_localhostrun()
        except FileNotFoundError as _e:
            print(f'[tunnel] cannot launch ({_e}); skipping this attempt.', flush=True); continue
        _routed = _bring_up(_p, _h)
        if _routed:
            active_proc, public_url = _p, _routed; break
        try: _p.terminate()
        except Exception: pass
        time.sleep(2)

    print('\n' + '=' * 70)
    if public_url:
        print('  PASTE THIS INTO THE APP  ->  Settings -> FLUX server URL:')
        print(f'  {public_url}')
    else:
        print('  FAILED: no working public tunnel after all attempts. The local server is fine,')
        print('  but nothing outside can reach it - retry "Start & connect" from the app.')
    print('=' * 70, flush=True)

    if public_url and active_proc:
        # ── Idle-shutdown watchdog ──────────────────────────────────────
        # After IDLE_SHUTDOWN_MIN minutes with no ESTABLISHED connection to the server port, stop the
        # tunnel so a forgotten run stops burning GPU quota. App polling / liveness counts as activity.
        IDLE_SHUTDOWN_MIN = 15
        def _idle_watchdog():
            _port_hex = ':%04X' % PORT
            _last = time.time()
            while True:
                time.sleep(30)
                _active = False
                for _tbl in ('/proc/net/tcp', '/proc/net/tcp6'):
                    try:
                        with open(_tbl) as _f:
                            for _l in _f.readlines()[1:]:
                                _q = _l.split()
                                if _q[1].endswith(_port_hex) and _q[3] == '01':
                                    _active = True; break
                    except OSError:
                        _active = True
                    if _active: break
                if _active:
                    _last = time.time()
                elif time.time() - _last > IDLE_SHUTDOWN_MIN * 60:
                    print(f'[watchdog] No requests for {IDLE_SHUTDOWN_MIN} min - shutting down to save GPU quota.', flush=True)
                    try: active_proc.terminate()
                    except Exception: pass
                    return
        threading.Thread(target=_idle_watchdog, daemon=True).start()
        print(f'Keep this cell running. Idle watchdog armed: auto-stops after {IDLE_SHUTDOWN_MIN} min idle.', flush=True)
        active_proc.wait()